# Compare three director-color schemes

This notebook compares the same three director-to-color maps in two ways:

1. on the director color sphere itself;
2. on the bundled `Q_example_workflow.npy` field, using the same practical visualization recipe as `tutorials/quick_visualize_q.ipynb`.

The three schemes are:

- original Nematics3D `n_color_immerse()`;
- previous Stage-I sRGB Pareto candidate, `director_color_pareto_034`;
- selected OKLab Stage-I Pareto knee, `director_color_pareto_oklab_043`.

For each comparison, geometry and visualization parameters are kept fixed. Only the director color function changes.

In [1]:
from pathlib import Path
import sys

import numpy as np

def find_repo_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / "example" / "data" / "Q_example_workflow.npy").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository root from the current working directory."
    )

REPO_ROOT = find_repo_root()

try:
    import nematics3d as n3d
except ModuleNotFoundError:
    sys.path.insert(0, str(REPO_ROOT / "src"))
    import nematics3d as n3d

from nematics3d.field import n_color_immerse
from nematics3d.classes.q_field_object import QFieldObject
from nematics3d.classes.visual.plot_figure import PlotFigure
from nematics3d.classes.visual.plot_tube import OptsTube
from nematics3d.classes.visual.color import (
    director_color_pareto_034,
    director_color_pareto_oklab_043,
    plot_director_color_sphere,
)
from nematics3d.quick import (
    _auto_quick_Q_visual_params,
    _resolve_director_spacing_level,
)

DATA_PATH = REPO_ROOT / "example" / "data" / "Q_example_workflow.npy"
Q_data = np.load(DATA_PATH)
Q_data.shape

(200, 100, 100, 5)

## Part I — color spheres

These three plots show the raw geometry of the three color maps on the director sphere.

### 1. Original Nematics3D scheme

In [2]:
scene_nematics3d = plot_director_color_sphere(
    n_color_immerse,
    figure_size=(1000, 1000),
)
scene_nematics3d

{'figure': PlotFigure('director_color_sphere')
 RegistryBase('figure glyph registry')
 0:       director_color_sphere
 1:       director_color_axes  
 ScalarBarRegistry('scalar bars manager')
 <empty registry>,
 'surface': PlotPolyData('director_color_sphere'),
 'axes': PlotVector('director_color_axes')}

### 2. Previous sRGB Pareto candidate ($J_{\mathrm{norm}}=0.34$)

In [3]:
scene_srgb_pareto = plot_director_color_sphere(
    director_color_pareto_034,
    figure_size=(1000, 1000),
)
scene_srgb_pareto

{'figure': PlotFigure('director_color_sphere')
 RegistryBase('figure glyph registry')
 0:       director_color_sphere
 1:       director_color_axes  
 ScalarBarRegistry('scalar bars manager')
 <empty registry>,
 'surface': PlotPolyData('director_color_sphere'),
 'axes': PlotVector('director_color_axes')}

### 3. OKLab Pareto knee ($J_{\mathrm{loc}}^{\mathrm{OKLab}}\approx0.43$)

In [4]:
scene_oklab_pareto = plot_director_color_sphere(
    director_color_pareto_oklab_043,
    figure_size=(1000, 1000),
)
scene_oklab_pareto

{'figure': PlotFigure('director_color_sphere')
 RegistryBase('figure glyph registry')
 0:       director_color_sphere
 1:       director_color_axes  
 ScalarBarRegistry('scalar bars manager')
 <empty registry>,
 'surface': PlotPolyData('director_color_sphere'),
 'axes': PlotVector('director_color_axes')}

## Part II — practical Q-field comparison

Now use the same bundled Q-tensor data and essentially the same construction as `quick_visualize_q()`.

To make the comparison clean, the Q field is initialized and its disclination lines are smoothed only once. Each of the three figures then receives the same line geometry, box extent, director-plane position, spacing, rod length, and rod radius. The only changed parameter is `n_color`.

The director plane uses the same default `grid_normal=(0,0,1)` and `director_spacing="medium"` as `quick_visualize_q()`.

In [5]:
grid_normal = (0, 0, 1)
director_spacing = "medium"

params = _auto_quick_Q_visual_params(Q_data, grid_normal)
director_spacing_config = _resolve_director_spacing_level(director_spacing)

Q_obj = QFieldObject(
    Q=Q_data,
    name="director-color-comparison",
    default_miminum_line_length_smooth=params["smooth_min_line_length"],
    default_smooth_window_length=params["smooth_window_length"],
    default_miminum_line_length_visual=params["visual_min_line_length"],
)

Q_obj.act_lines_smooth(
    min_line_length=params["smooth_min_line_length"],
    window_length=params["smooth_window_length"],
)

[PROGRESS]
    <QFieldObject.__init__> 
    Start to initialize Q tensor `director-color-comparison`.
[PROGRESS]
    <QFieldObject.__init__> 
    Start defect analysis as detecting defects and classifying them into distinct lines for Q tensor `director-color-comparison` 
    This operation might take a while.
    You can disable this automatic operation by setting is_detect_defects=False and is_classify_lines=False when initializing the Q tensor.
[INFO]
        <QFieldObject[name='director-color-comparison'].act_defect_detect> 
        1270 defects are found.
[INFO]
        <QFieldObject[name='director-color-comparison'].act_lines_classify> 
        8 lines are found.
[PROGRESS]
    <QFieldObject.__init__> 
    Defect analysis is finished, with 0.06 s
[INFO]
    <QFieldObject[name='director-color-comparison'].act_lines_smooth> 
    There are 8 disclination lines in total, with 7 lines are smoothed.
    The smoothing window length is: 41


In [6]:
def make_qfield_color_comparison_figure(color_func, label):
    figure = PlotFigure()

    Q_obj.act_visualize_disclination_lines(
        figure=figure,
        is_extent=False,
        min_line_length=params["visual_min_line_length"],
        line_radius=params["line_radius"],
    )

    Q_obj.calc_bounds.act_visualize(
        figure=figure,
        opts=OptsTube(radius=params["extent_radius"]),
        is_reset_camera=False,
    )

    Q_obj.act_visualize_n_plane(
        figure=figure,
        is_extent=False,
        grid_normal=grid_normal,
        grid_spacing=(
            params["grid_spacing"]
            * director_spacing_config["grid_spacing_scale"]
        ),
        grid_size=params["grid_size"],
        grid_origin=params["grid_origin"],
        n_length=(
            params["n_length"]
            * director_spacing_config["n_length_scale"]
        ),
        n_radius=(
            params["n_radius"]
            * director_spacing_config["n_radius_scale"]
        ),
        n_color=color_func,
        plane_name=f"n-plane-{label}",
    )

    return figure

### 1. Original Nematics3D colors on the Q field

In [7]:
figure_q_original = make_qfield_color_comparison_figure(
    n_color_immerse,
    "original",
)
figure_q_original

PlotFigure('disclination lines')
RegistryBase('figure glyph registry')
 0:       disclination line 0 smooth_version 0                            
 1:       disclination line 1 smooth_version 0                            
 2:       disclination line 2 smooth_version 0                            
 3:       disclination line 3 smooth_version 0                            
 4:       disclination line 4 smooth_version 0                            
 5:       disclination line 5 smooth_version 0                            
 6:       disclination line 6 smooth_version 0                            
 7:       Bounds of grid field dataset 'director-color-comparison dataset'
 8:       n bulk of plane 'n-plane-original'                              
 9:       n near defect of plane 'n-plane-original'                       
10:       defects of plane 'n-plane-original'                             
ScalarBarRegistry('scalar bars manager')
<empty registry>

### 2. Previous sRGB Pareto colors on the Q field

In [8]:
figure_q_srgb = make_qfield_color_comparison_figure(
    director_color_pareto_034,
    "srgb-pareto-034",
)
figure_q_srgb

[WARNING]
                <QFieldObject[name='director-color-comparison'] -> _helper_check_name> 
                >>> 'disclination lines' already exists in Registry 'figures'! Renamed to 'disclination lines_1'.
                Current warning call: D:\Document\GitHub\Nematics3D\src\nematics3d\classes\registry_base.py:111
                Caller: D:\Document\GitHub\Nematics3D\src\nematics3d\classes\registry_base.py:155
                code: name = self._helper_check_name(term.name)


PlotFigure('disclination lines_1')
RegistryBase('figure glyph registry')
 0:       disclination line 0 smooth_version 1                            
 1:       disclination line 1 smooth_version 1                            
 2:       disclination line 2 smooth_version 1                            
 3:       disclination line 3 smooth_version 1                            
 4:       disclination line 4 smooth_version 1                            
 5:       disclination line 5 smooth_version 1                            
 6:       disclination line 6 smooth_version 1                            
 7:       Bounds of grid field dataset 'director-color-comparison dataset'
 8:       n bulk of plane 'n-plane-srgb-pareto-034'                       
 9:       n near defect of plane 'n-plane-srgb-pareto-034'                
10:       defects of plane 'n-plane-srgb-pareto-034'                      
ScalarBarRegistry('scalar bars manager')
<empty registry>

### 3. OKLab Pareto-knee colors on the Q field

In [9]:
figure_q_oklab = make_qfield_color_comparison_figure(
    director_color_pareto_oklab_043,
    "oklab-pareto-043",
)
figure_q_oklab

[WARNING]
                <QFieldObject[name='director-color-comparison'] -> _helper_check_name> 
                >>> 'disclination lines' already exists in Registry 'figures'! Renamed to 'disclination lines_2'.
                Current warning call: D:\Document\GitHub\Nematics3D\src\nematics3d\classes\registry_base.py:111
                Caller: D:\Document\GitHub\Nematics3D\src\nematics3d\classes\registry_base.py:155
                code: name = self._helper_check_name(term.name)


PlotFigure('disclination lines_2')
RegistryBase('figure glyph registry')
 0:       disclination line 0 smooth_version 2                            
 1:       disclination line 1 smooth_version 2                            
 2:       disclination line 2 smooth_version 2                            
 3:       disclination line 3 smooth_version 2                            
 4:       disclination line 4 smooth_version 2                            
 5:       disclination line 5 smooth_version 2                            
 6:       disclination line 6 smooth_version 2                            
 7:       Bounds of grid field dataset 'director-color-comparison dataset'
 8:       n bulk of plane 'n-plane-oklab-pareto-043'                      
 9:       n near defect of plane 'n-plane-oklab-pareto-043'               
10:       defects of plane 'n-plane-oklab-pareto-043'                     
ScalarBarRegistry('scalar bars manager')
<empty registry>

The second set of three figures is the practical comparison to inspect most closely. Since the Q field, camera construction, defects, plane geometry, and rod geometry are held fixed, visible differences among these figures come from the director color mapping rather than from a changed dataset or sampling choice.